# Case Study 01 — Credit Scoring: Feature Engineering

**Erick Condoy** · Economist (UNL) · Quant Researcher  
Repository: [credit-risk-lab](https://github.com/EryckFCS/credit-risk-lab)

---

## Objective

Transform raw UCI features into model-ready inputs using **Weight of Evidence (WoE)** encoding — the industry-standard method for credit scorecard development (Siddiqi, 2006; Anderson, 2007).

| Section | Content |
|---------|---------|
| 1 | Load & Reproduce EDA State |
| 2 | Categorical Encoding — Remap Undocumented Codes |
| 3 | Manual Feature Engineering (derived ratios) |
| 4 | WoE Binning — All Features |
| 5 | IV Table — Final Feature Selection |
| 6 | WoE Transformation — Train/Test Split |
| 7 | Correlation Check on WoE Features |
| 8 | Save Artefacts |

---

### Theoretical Background

**Weight of Evidence** for bin $i$:
$$\text{WoE}_i = \ln\left(\frac{\text{Dist\_Bad}_i}{\text{Dist\_Good}_i}\right)$$

**Information Value:**
$$\text{IV} = \sum_i (\text{Dist\_Bad}_i - \text{Dist\_Good}_i) \times \text{WoE}_i$$

WoE encoding maps each bin to a monotonic log-odds scale — linear models (Logistic Regression) can then capture the relationship directly without polynomial terms.

In [ ]:
# ── Environment ────────────────────────────────────────────────────────────
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 40)

REPO_ROOT = Path().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_DIR    = Path('../data')
REPORTS_DIR = Path('../reports')
ARTEFACTS   = Path('../artefacts')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
ARTEFACTS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#dcd9d5', 'axes.labelcolor': '#28251d',
    'xtick.color': '#7a7974', 'ytick.color': '#7a7974',
    'font.family': 'sans-serif', 'font.size': 10,
    'axes.titlesize': 11, 'axes.titleweight': 'semibold',
    'figure.dpi': 120,
})
TEAL='#01696f'; MAROON='#a12c7b'; GRAY='#bab9b4'; GREEN='#437a22'; ORANGE='#964219'
print(f'Python {sys.version}')
print(f'pandas {pd.__version__} | numpy {np.__version__}')

## 1. Load & Reproduce EDA State

In [ ]:
def _find_dataset(data_dir: Path) -> Path:
    for pattern in ('*.parquet', '*.csv', '*.xlsx', '*.xls'):
        hits = list(data_dir.glob(pattern))
        if hits: return hits[0]
    raise FileNotFoundError('Dataset not found. Run: bash data/download.sh')

fpath = _find_dataset(DATA_DIR)
raw = pd.read_parquet(fpath) if fpath.suffix == '.parquet' else (
      pd.read_excel(fpath, header=1) if fpath.suffix in ('.xls','.xlsx') else pd.read_csv(fpath))

raw.columns = raw.columns.str.strip().str.lower().str.replace(' ','_').str.replace('.','_')
target_raw  = [c for c in raw.columns if 'default' in c][0]
df = raw.rename(columns={target_raw: 'default', 'id': 'client_id'} if 'id' in raw.columns
               else {target_raw: 'default'})

TARGET   = 'default'
ID_COL   = 'client_id' if 'client_id' in df.columns else None
FEATURES = [c for c in df.columns if c not in ([TARGET] + ([ID_COL] if ID_COL else []))]

print(f'Loaded {len(df):,} rows × {len(FEATURES)} features')
print(f'Default rate: {df[TARGET].mean():.2%}')

## 2. Categorical Encoding — Remap Undocumented Codes

In [ ]:
# UCI dataset has undocumented codes 0, 5, 6 in EDUCATION and 0 in MARRIAGE.
# Industry practice: merge them into 'Others' category (code 4 / 3 respectively).
df = df.copy()

if 'education' in df.columns:
    df['education'] = df['education'].apply(lambda x: x if x in [1,2,3,4] else 4)
    print('[OK] EDUCATION: codes 0,5,6 → 4 (Others)')

if 'marriage' in df.columns:
    df['marriage'] = df['marriage'].apply(lambda x: x if x in [1,2,3] else 3)
    print('[OK] MARRIAGE: code 0 → 3 (Others)')

# Verify
for col, valid in [('education',[1,2,3,4]),('marriage',[1,2,3]),('sex',[1,2])]:
    if col in df.columns:
        invalid_count = (~df[col].isin(valid)).sum()
        status = 'OK' if invalid_count == 0 else f'WARN: {invalid_count} invalid'
        print(f'  [{status}] {col.upper()}')

## 3. Manual Feature Engineering (Derived Ratios)

In [ ]:
bill_cols    = sorted([c for c in df.columns if c.startswith('bill_amt')])
pay_amt_cols = sorted([c for c in df.columns if c.startswith('pay_amt')])
pay_stat_cols= sorted([c for c in df.columns if c.startswith('pay_') and c[4:].isdigit()])[:6]

# --- Ratio features ---
df['avg_bill']         = df[bill_cols].mean(axis=1)
df['avg_pay_amt']      = df[pay_amt_cols].mean(axis=1)
df['utilization']      = np.clip(df['avg_bill'] / df['limit_bal'].replace(0, np.nan), 0, 5).fillna(0)
df['pay_ratio']        = np.clip(df['avg_pay_amt'] / df['avg_bill'].replace(0, np.nan), 0, 5).fillna(0)

# --- Delinquency count: months with payment delay >= 1 ---
if pay_stat_cols:
    df['n_delinquent_months'] = (df[pay_stat_cols] >= 1).sum(axis=1)
    df['max_delay']           = df[pay_stat_cols].max(axis=1)
    df['pay_trend']           = df[pay_stat_cols[0]] - df[pay_stat_cols[-1]]  # recent minus oldest

# --- Bill trend: is the balance growing? ---
if len(bill_cols) >= 2:
    df['bill_trend'] = df[bill_cols[0]] - df[bill_cols[-1]]  # most recent - oldest

# --- Limit utilisation buckets (manual binning) ---
df['limit_band'] = pd.cut(df['limit_bal'],
    bins=[0, 50_000, 100_000, 200_000, 500_000, np.inf],
    labels=['micro','low','mid','high','premium'])

DERIVED = ['utilization','pay_ratio','n_delinquent_months','max_delay',
           'pay_trend','bill_trend','limit_band']
DERIVED = [c for c in DERIVED if c in df.columns]

print(f'Derived features added: {DERIVED}')
print(df[DERIVED].describe().T[['mean','std','min','max']])

## 4. WoE Binning — All Features

In [ ]:
def woe_bin(series: pd.Series, target: pd.Series, bins: int = 10) -> pd.DataFrame:
    """
    Compute WoE and IV table for a single feature.
    Returns DataFrame with columns: bin, n, n_bad, n_good, dist_bad, dist_good, woe, iv_bin.
    """
    df_w = pd.DataFrame({'x': series, 'y': target})
    if series.dtype in ['object', 'category'] or series.nunique() <= bins:
        df_w['bin'] = series.astype(str)
    else:
        df_w['bin'] = pd.qcut(series, q=bins, duplicates='drop').astype(str)

    total_bad  = max((target == 1).sum(), 1)
    total_good = max((target == 0).sum(), 1)

    tbl = df_w.groupby('bin')['y'].agg(
        n='count',
        n_bad=lambda x: (x==1).sum(),
        n_good=lambda x: (x==0).sum()
    ).reset_index()
    tbl['dist_bad']  = tbl['n_bad']  / total_bad
    tbl['dist_good'] = tbl['n_good'] / total_good
    tbl['pct_total'] = tbl['n'] / len(series)
    tbl['default_rate'] = tbl['n_bad'] / tbl['n']
    # Exclude zero-distribution bins to avoid log(0)
    mask = (tbl['dist_bad'] > 0) & (tbl['dist_good'] > 0)
    tbl.loc[mask,  'woe']    = np.log(tbl.loc[mask,'dist_bad'] / tbl.loc[mask,'dist_good'])
    tbl.loc[~mask, 'woe']   = 0.0
    tbl['iv_bin'] = (tbl['dist_bad'] - tbl['dist_good']) * tbl['woe']
    return tbl


def compute_iv_from_table(tbl: pd.DataFrame) -> float:
    return tbl['iv_bin'].sum()


ALL_FEATURES = FEATURES + DERIVED
ALL_FEATURES = [c for c in dict.fromkeys(ALL_FEATURES) if c in df.columns]  # dedup, preserve order

woe_tables: dict[str, pd.DataFrame] = {}
iv_scores:  dict[str, float]         = {}

for col in ALL_FEATURES:
    tbl = woe_bin(df[col], df[TARGET])
    woe_tables[col] = tbl
    iv_scores[col]  = compute_iv_from_table(tbl)

iv_df = (
    pd.DataFrame.from_dict(iv_scores, orient='index', columns=['IV'])
    .sort_values('IV', ascending=False)
    .assign(
        strength=lambda x: x['IV'].map(lambda v:
            'Useless'    if v < 0.02 else
            'Weak'       if v < 0.10 else
            'Medium'     if v < 0.30 else
            'Strong'     if v < 0.50 else 'Very Strong'),
        source=lambda x: x.index.map(lambda c: 'derived' if c in DERIVED else 'original')
    )
)

print(f'Total features evaluated: {len(iv_df)}')
print(f'Medium+ (IV > 0.10): {(iv_df.IV > 0.10).sum()}')
print(f'Strong+ (IV > 0.30): {(iv_df.IV > 0.30).sum()}')
display(iv_df.style.format({'IV': '{:.4f}'})
        .bar(subset=['IV'], color=TEAL)
        .applymap(lambda v: f'color: {TEAL}' if v=='original' else f'color: {ORANGE}', subset=['source']))

## 5. IV Table — Final Feature Selection

In [ ]:
IV_THRESHOLD = 0.02  # Siddiqi: drop 'Useless' features
SELECTED = iv_df[iv_df['IV'] >= IV_THRESHOLD].index.tolist()
DROPPED  = iv_df[iv_df['IV'] <  IV_THRESHOLD].index.tolist()

print(f'Selected features (IV >= {IV_THRESHOLD}): {len(SELECTED)}')
print(f'Dropped features  (IV <  {IV_THRESHOLD}): {len(DROPPED)}')
if DROPPED:
    print(f'  Dropped: {DROPPED}')

# ---- WoE Profile Plot for top-10 features ----
top10 = iv_df.head(10).index.tolist()
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()
for ax, feat in zip(axes, top10):
    tbl = woe_tables[feat]
    iv_val = iv_scores[feat]
    colors = [MAROON if w > 0 else TEAL for w in tbl['woe']]
    ax.bar(range(len(tbl)), tbl['woe'], color=colors, edgecolor='none', width=0.7)
    ax.axhline(0, color='#dcd9d5', lw=1)
    ax.set_title(f'{feat}\nIV={iv_val:.3f}', fontsize=9)
    ax.set_xticks(range(len(tbl)))
    ax.set_xticklabels(tbl['bin'], rotation=45, ha='right', fontsize=6)
    ax.set_ylabel('WoE', fontsize=8)
plt.suptitle('WoE Profiles — Top 10 Predictors', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(REPORTS_DIR / '08_woe_profiles_top10.png', bbox_inches='tight', dpi=150)
plt.show()

# ---- IV ranking bar chart ----
strength_colors = {
    'Very Strong':'#01696f','Strong':'#4f98a3','Medium':'#d19900',
    'Weak':'#da7101','Useless':'#dcd9d5'
}
iv_plot = iv_df[iv_df['IV'] >= IV_THRESHOLD]
bar_colors = iv_plot['strength'].map(strength_colors).values
fig, ax = plt.subplots(figsize=(9, max(5, len(iv_plot)*0.38)))
ax.barh(iv_plot.index[::-1], iv_plot['IV'][::-1], color=bar_colors[::-1], edgecolor='none')
ax.axvline(0.10, color=ORANGE, linestyle='--', lw=1.2, label='Medium (0.10)')
ax.axvline(0.30, color=TEAL,   linestyle='--', lw=1.2, label='Strong (0.30)')
ax.set_xlabel('Information Value (IV)')
ax.set_title('Final IV Ranking — Selected Features', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(REPORTS_DIR / '09_iv_ranking_final.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. WoE Transformation — Train/Test Split

In [ ]:
# ---- Build WoE encoder from training set only (no leakage) ----

X = df[SELECTED].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Train default rate: {y_train.mean():.2%}  |  Test default rate: {y_test.mean():.2%}')


def build_woe_map(X_tr: pd.DataFrame, y_tr: pd.Series,
                  features: list[str], bins: int = 10) -> dict[str, dict]:
    """
    Fit WoE bins on training set. Returns a mapping {feature: {bin_label: woe_value}}.
    """
    woe_map = {}
    for col in features:
        tbl = woe_bin(X_tr[col], y_tr, bins=bins)
        woe_map[col] = dict(zip(tbl['bin'], tbl['woe']))
    return woe_map


def apply_woe(X: pd.DataFrame, woe_map: dict[str, dict],
              bins: int = 10) -> pd.DataFrame:
    """
    Apply pre-fitted WoE map to any split. Unknown bins → WoE = 0 (neutral).
    """
    X_woe = pd.DataFrame(index=X.index)
    for col, mapping in woe_map.items():
        if col not in X.columns:
            continue
        s = X[col]
        if s.dtype in ['object','category'] or s.nunique() <= bins:
            binned = s.astype(str)
        else:
            # Compute cut points from training distribution stored as string labels
            # Re-bin using the original boundaries inferred from quantile cut
            try:
                binned = pd.qcut(s, q=bins, duplicates='drop').astype(str)
            except ValueError:
                binned = s.astype(str)
        X_woe[f'{col}_woe'] = binned.map(mapping).fillna(0.0)
    return X_woe


woe_map = build_woe_map(X_train, y_train, SELECTED)

X_train_woe = apply_woe(X_train, woe_map)
X_test_woe  = apply_woe(X_test,  woe_map)

print(f'WoE feature matrix shape — train: {X_train_woe.shape}, test: {X_test_woe.shape}')
print(f'Unknown bins → filled with WoE=0.0 (train median)')
X_train_woe.head(3)

## 7. Correlation Check on WoE Features

In [ ]:
# Drop features with |Pearson r| > 0.75 with another feature (multicollinearity)
corr_woe = X_train_woe.corr().abs()
upper = corr_woe.where(np.triu(np.ones(corr_woe.shape), k=1).astype(bool))
high_corr_pairs = [(col, row) for col in upper.columns
                   for row in upper.index if upper.loc[row, col] > 0.75]

if high_corr_pairs:
    print(f'[WARN] High-correlation pairs (|r|>0.75): {len(high_corr_pairs)}')
    for a, b in high_corr_pairs:
        print(f'  {a} ↔ {b}  r={corr_woe.loc[b,a]:.3f}')
    # Drop the feature with lower IV from each pair
    to_drop = set()
    for a, b in high_corr_pairs:
        feat_a = a.replace('_woe','')
        feat_b = b.replace('_woe','')
        drop = feat_b if iv_scores.get(feat_a,0) >= iv_scores.get(feat_b,0) else feat_a
        to_drop.add(drop + '_woe')
    print(f'  → Dropping lower-IV features: {to_drop}')
    X_train_woe = X_train_woe.drop(columns=list(to_drop), errors='ignore')
    X_test_woe  = X_test_woe.drop(columns=list(to_drop),  errors='ignore')
else:
    print('[OK] No high-correlation pairs detected (threshold |r|>0.75).')

# Heatmap
fig, ax = plt.subplots(figsize=(min(18, len(X_train_woe.columns)*0.7+2),
                                min(16, len(X_train_woe.columns)*0.7+2)))
mask = np.triu(np.ones_like(X_train_woe.corr(), dtype=bool))
sns.heatmap(X_train_woe.corr(), mask=mask,
            cmap=sns.diverging_palette(220,20,as_cmap=True),
            center=0, vmin=-1, vmax=1, annot=True, fmt='.2f',
            annot_kws={'size':6}, linewidths=0.2, ax=ax)
ax.set_title('WoE Feature Correlation Matrix (Train)', fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_DIR / '10_woe_correlation_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

FINAL_FEATURES = list(X_train_woe.columns)
print(f'\nFinal WoE feature count: {len(FINAL_FEATURES)}')

## 8. Save Artefacts

In [ ]:
import json, pickle

# ---- Save WoE map (JSON — human-readable, version-controllable) ----
woe_map_path = ARTEFACTS / 'woe_map.json'
with open(woe_map_path, 'w') as f:
    json.dump(woe_map, f, indent=2)
print(f'[saved] woe_map.json → {woe_map_path}')

# ---- Save train/test WoE matrices ----
X_train_woe.to_parquet(ARTEFACTS / 'X_train_woe.parquet', index=True)
X_test_woe.to_parquet( ARTEFACTS / 'X_test_woe.parquet',  index=True)
y_train.to_frame().to_parquet(ARTEFACTS / 'y_train.parquet', index=True)
y_test.to_frame().to_parquet( ARTEFACTS / 'y_test.parquet',  index=True)
print(f'[saved] X_train_woe, X_test_woe, y_train, y_test → {ARTEFACTS}')

# ---- Save IV table ----
iv_df.to_csv(ARTEFACTS / 'iv_table.csv')
print(f'[saved] iv_table.csv → {ARTEFACTS}')

# ---- Save selected feature list ----
with open(ARTEFACTS / 'selected_features.json', 'w') as f:
    json.dump({'woe_features': FINAL_FEATURES, 'original_selected': SELECTED}, f, indent=2)
print(f'[saved] selected_features.json → {ARTEFACTS}')

print()
print('=== Feature Engineering Summary ===')
print(f'  Original features evaluated : {len(ALL_FEATURES)}')
print(f'  Derived features engineered : {len(DERIVED)}')
print(f'  Selected after IV filter    : {len(SELECTED)}')
print(f'  Final WoE features          : {len(FINAL_FEATURES)}')
print(f'  Train size                  : {len(X_train_woe):,}')
print(f'  Test size                   : {len(X_test_woe):,}')
print()
print('Next notebook: 03_modeling_logistic_regression.ipynb')